In [1]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 

ROOT_DIR = '../..'
sys.path.append(f'{ROOT_DIR}/src')



import openai
import os
import json

def load_api_keys(root_dir):
    import json
    with open(f"{root_dir}/API_KEYS2.json", "r") as file:
        api_keys = json.load(file)
    os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
    os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
    os.environ['LLMS_CACHE_PATH'] = "massmaps_qwen.cache"
    # os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.path.join(root_dir, api_keys['GOOGLE_APPLICATION_CREDENTIALS'])
    os.environ['CACHE_DIR'] = os.path.join(root_dir, 'cache_dir3')
    return api_keys

load_api_keys(ROOT_DIR);

# Cholec

In [2]:
import importlib
import sys; sys.path.append("../src")
import cholec
importlib.reload(cholec)
from cholec import get_llm_generated_answer
from cholec import CholecExample, CholecDataset, load_model #, items_to_examples
from cholec import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_score, group_claims_by_category, make_alignment_matrix, categories_list

In [3]:
test_dataset = CholecDataset(split="test")

In [4]:
from tqdm.auto import tqdm
import json

In [5]:
# model = 'gpt-4o'
models = [
    "gpt-5.2-pro-2025-12-11",
    "gpt-5-mini-2025-08-07",
    "claude-opus-4-5-20251101",
    "claude-haiku-4-5-20251001",
    "gemini-2.5-pro",
    "gemini-2.5-flash"
]

eval_model = load_model("Qwen/Qwen2.5-VL-7B-Instruct", max_model_len=16384)
eval_model_name = 'qwen2.5-vl'


INFO 01-28 03:34:43 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 01-28 03:34:44 [__init__.py:239] Automatically detected platform cuda.
Loading Qwen-VL with vLLM: Qwen/Qwen2.5-VL-7B-Instruct
INFO 01-28 03:34:53 [config.py:717] This model supports multiple tasks: {'embed', 'generate', 'classify', 'score', 'reward'}. Defaulting to 'generate'.
INFO 01-28 03:34:53 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 01-28 03:34:55 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen2.5-VL-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-VL-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


WARNING 01-28 03:35:01 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
INFO 01-28 03:35:01 [gpu_model_runner.py:1329] Starting to load model Qwen/Qwen2.5-VL-7B-Instruct...
WARNING 01-28 03:35:01 [vision.py:93] Current `vllm-flash-attn` has a bug inside vision module, so we use xformers backend instead. You can run `pip install flash-attn` to use flash-attention backend.
INFO 01-28 03:35:01 [config.py:3614] cudagraph sizes specified by model runner [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 264, 272, 280, 288, 296, 304, 312, 320, 328, 336, 344, 352, 360, 368, 376, 384, 392, 400, 408, 416, 424, 432, 440, 448, 456, 464, 472, 480, 488, 496, 504, 512] is overridden by config [512, 384, 256, 128, 4, 2, 1, 392, 264, 136, 8, 400, 272, 144, 16, 408

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 01-28 03:35:05 [loader.py:458] Loading weights took 3.66 seconds
INFO 01-28 03:35:05 [gpu_model_runner.py:1347] Model loading took 15.6277 GiB and 4.286312 seconds
INFO 01-28 03:35:08 [gpu_model_runner.py:1620] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
INFO 01-28 03:35:16 [backends.py:420] Using cache directory: /home/runai-home/.cache/vllm/torch_compile_cache/cca45916b7/rank_0_0 for vLLM's torch.compile
INFO 01-28 03:35:16 [backends.py:430] Dynamo bytecode transform time: 4.78 s
INFO 01-28 03:35:20 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 3.040 s
INFO 01-28 03:35:25 [monitor.py:33] torch.compile takes 4.78 s in total
INFO 01-28 03:35:26 [kv_cache_utils.py:634] GPU KV cache size: 915,984 tokens
INFO 01-28 03:35:26 [kv_cache_utils.py:637] Maximum concurrency for 16,384 tokens per request: 55.91x
INFO 01-28 03:35:57 [gpu_model_runner.py:1686] Graph cap

In [6]:
methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

In [7]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
import json
import copy
from tqdm.auto import tqdm

# for model in models:
#     print(f"=== Using model {model} ===")
#     for method in methods:
        # print(f"=== Using method {method} ===")

model = models[0]
method = methods[0]
    
load_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}.json')
save_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}_{eval_model_name}.json')

with open(load_path) as input_file:
    results = json.load(input_file)
    
results[0].keys()

dict_keys(['id', 'true_safe_list', 'true_unsafe_list', 'llm_raw_output', 'llm_explanation', 'llm_safe_list', 'llm_unsafe_list', 'all_claims', 'relevant_claims', 'alignable_claims', 'aligned_category_ids', 'alignment_scores', 'alignment_reasonings', 'final_alignment_score', 'safe_iou', 'unsafe_iou', 'claims_by_category', 'category_alignment_scores', 'category_alignment_reasonings', '_filename'])

In [9]:
results[0]['id'], results[1]['id'], results[2]['id']

('M2CCAI2016_video107_001.png',
 'M2CCAI2016_video107_002.png',
 'M2CCAI2016_video113_001.png')

In [10]:
test_dataset.__dict__.keys()

dict_keys(['dataset', 'image_size', 'preprocess_image', 'preprocess_labels'])

In [11]:
id2idx_mapping = {
    test_dataset.dataset[i]['id']: i
    for i in range(len(test_dataset.dataset))
}

In [12]:
test_dataset[id2idx_mapping[results[0]['id']]]['image'];

In [13]:
test_dataset.dataset[0]['id']

'M2CCAI2016_video107_001.png'

In [16]:
import json
import copy
import numpy as np
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        load_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/cholec_{model}_{eval_model_name}.json')

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = len(results)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            image = test_dataset[id2idx_mapping[result['id']]]['image']
            
            example = CholecExample(
                id=result['id'],
                image = image,
                true_safe_list=result['true_safe_list'],
                true_unsafe_list=result['true_unsafe_list'],
                llm_raw_output=result['llm_raw_output'],
                llm_explanation=result['llm_explanation'],
                llm_safe_list=result['llm_safe_list'],
                llm_unsafe_list=result['llm_unsafe_list'],
            )
            example.safe_iou = result['safe_iou']
            example.unsafe_iou = result['unsafe_iou']
            
            
            # isolate individual features
            all_claims = isolate_individual_features(example.llm_explanation, model=eval_model)
            if all_claims is None:
                continue
            example.all_claims = [claim.strip() for claim in all_claims]

            # distill relevant features
            relevant_claims = distill_relevant_features(
                example.image, 
                example.all_claims,
                model=eval_model
            )
            example.relevant_claims = relevant_claims

            # calculate expert alignment scores
#             align_infos = calculate_expert_alignment_scores(
#                 example.relevant_claims, 
#                 eval_model,
#             )

#             alignable_claims = [info["Claim"] for info in align_infos]
#             alignment_categories = [info["Category"] for info in align_infos]
#             aligned_category_ids = [info["Category ID"] for info in align_infos]
#             alignment_scores = [info["Alignment"] for info in align_infos]
#             alignment_raws = [info["Alignment Raw"] for info in align_infos]
#             alignment_reasonings = [info["Reasoning"] for info in align_infos]
            
#             example.alignable_claims = alignable_claims
#             example.alignment_categories = alignment_categories
#             example.aligned_category_ids = aligned_category_ids
#             example.alignment_scores = alignment_scores
#             example.alignment_raws = alignment_raws
#             example.alignment_reasonings = alignment_reasonings
            
#             # Non-alignable claims are given a score of 0.0
#             if len(align_infos) > 0:
#                 example.final_alignment_score = sum(info["Alignment"] for info in align_infos) / len(example.all_claims)
#             else:
#                 example.final_alignment_score = 0.0
    
            claims_by_category, category_alignment_scores, category_alignment_reasonings = calculate_expert_alignment_score(
                relevant_claims, 
                model=eval_model,
                # verbose=True
            )

            example.claims_by_category = claims_by_category
            example.category_alignment_scores = category_alignment_scores
            example.category_alignment_reasonings = category_alignment_reasonings

            alignment_matrix = make_alignment_matrix(
                example.all_claims,
                claims_by_category,
                category_alignment_scores
            )

            final_alignment_score = alignment_matrix.max(axis=-1).mean()
            if np.isnan(final_alignment_score):
                print(f'example {idx} final_alignment_score is NaN')
            example.final_alignment_score = final_alignment_score
    
            # save
            save_dict = {}
            for k, v in example.__dict__.items():
                if not isinstance(v, torch.Tensor):
                    save_dict[k] = v # if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
            # with open(save_path, 'wt') as output_file:
            #     json.dump(save_dict, output_file)

            new_results.append(save_dict)


        with open(save_path, 'wt') as output_file:
            json.dump(new_results, output_file, indent=4)

=== Using model gpt-5.2-pro-2025-12-11 ===
=== Using method vanilla ===


  0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

KeyboardInterrupt: 

# Cholec

# Emotion